In [ ]:
import dai
import pandas as pd
import numpy as np
import math
import warnings
from datetime import datetime, timedelta

from bigmodule import M
from bigtrader.finance.commission import PerOrder
from bigquant import bigtrader, dai

In [ ]:
# 因子测试 初始化

from datetime import datetime, timedelta

# 策略超参
sd = '2024-08-19'
ed = datetime.now().date().strftime("%Y-%m-%d")
holding_day = 10
target_hold_count = 10

# 因子sql
strateyg_factor_sql = f"""
            pct_rank_by(date,total_market_cap) as feature1,
            m_avg(turn,20) as avg_turn,
            pct_rank_by(date,avg_turn) as feature2,
            0.8 *feature1 + (1 - 0.8) * feature2 as score
    """

# 经过风格因子的分析，这个小市值加换手率的策略, 还是基于小市值的策略，加入换手率的择时
# 策略sql
strategy_sql = f"""
            select
            date,
            instrument,
            {strateyg_factor_sql}
            from
            cn_stock_prefactors
            where
            is_risk_warning=0
            and
            list_days>365
            and
            pe_ttm>0
            and
            list_sector<3
        """




In [ ]:
def strategy_run(strategy_sql,sd,ed,holding_day,target_hold_count):
    def initialize(context: bigtrader.IContext):

        from bigtrader.finance.commission import PerOrder

        # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
        context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))

        context.sql =strategy_sql


        context.data =  dai.query(context.sql,filters={'date':[context.start_date,context.end_date]}).df().dropna()
        context.holding_days = holding_day
        context.target_hold_count = target_hold_count


    def befor_trading(context, data):
        pass
        
    def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):

        # 每 context.holding_days 个交易日调仓一次
        if context.trading_day_index % context.holding_days != 0:
            return

        # 获取当前日期
        current_date = data.current_dt.strftime("%Y-%m-%d")
        # 获取当日数据
        current_day_data = context.data[context.data["date"] == current_date]
        # 取前10只
        current_day_data.sort_values(by='score',inplace=True,ascending=True)
        current_day_data = current_day_data.head(context.target_hold_count)
        len_ = len(current_day_data)
        # 获取当日目标持有股票
        target_hold_instruments = set(current_day_data["instrument"])
        
        # 获取当前已持有股票
        current_hold_instruments = set(context.get_account_positions().keys())

        # 卖出不在目标持有列表中的股票
        for instrument in current_hold_instruments - target_hold_instruments:
            context.order_target_percent(instrument, 0)
            
        # 买入目标持有列表中的股票
        for instrument in target_hold_instruments - current_hold_instruments:
            context.order_target_percent(instrument, 1/len_)

    performance = bigtrader.run(
        market=bigtrader.Market.CN_STOCK,
        frequency=bigtrader.Frequency.DAILY,
        start_date=sd, 
        end_date=ed,  
        capital_base=3000000,     # 设置初始资金
        initialize=initialize,     # 传入初始化函数
        handle_data=handle_data,   # 传入数据处理函数
        before_trading_start = befor_trading,
        order_price_field_buy='open',
        order_price_field_sell='open',
    )

    # 渲染绩效报告，展示回测结果
    # performance.render()
    return performance

In [ ]:
performce = strategy_run(strategy_sql,sd,ed,holding_day,target_hold_count)

raw_perf = performce.raw_perf.reset_index(drop=True)
annual_return,sharpe_ratio,max_drawdown = calc_key_index(raw_perf[['date','portfolio_value']])
performce.render()
